# Stage 0 · CPU notebook

Ticket 01: CPU-only source readiness and durable status. No model execution is implemented here yet.
The two notebooks share saved plans under the same artifact root.

## 1. Storage and checkout
Mount Drive when in Colab; select the existing Stage 0 release checkout. CPU runtime is sufficient for all current cells.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Use the checkout opened for this notebook. Published releases pin this checkout.
REPO_DIR = Path('/content/spar-stage0') if Path('/content').is_dir() else Path.cwd()
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts') if Path('/content').is_dir() else REPO_DIR / 'artifacts'
STAGE0_NAME = 'stage0-geometry-v1'
PLAN = ARTIFACT_ROOT / 'plans' / STAGE0_NAME / 'stage0.plan.json'

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
if not (REPO_DIR / 'pilot_eval').is_dir():
    raise RuntimeError('Open the Stage 0 release checkout first; see docs/stage0-geometry.md. No GPU is needed.')

def run_stage0(operation, *arguments):
    command = [sys.executable, '-u', '-m', 'pilot_eval', 'stage0-' + operation,
               '--output-root', str(ARTIFACT_ROOT), *map(str, arguments)]
    return subprocess.run(command, cwd=REPO_DIR, check=True)


## 2. Select and prepare
Bind the SFT/GRPO prepared sources. Optional inputs remain distinguishable from verified endpoint adapters.

In [ ]:
# Select existing prepared products; paths are relative to ARTIFACT_ROOT.
SELECTION = ARTIFACT_ROOT / 'notebook' / STAGE0_NAME / 'source-selection.json'
settings = {
    'sources': {
        'sft': 'plans/pilot3-inputs-v1/activation.prepared.json',
        'grpo': 'plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json',
        'random': 'plans/pilot4-grpo-fp32-v1-random-writes-v1/activation.prepared.json',
    },
    # Optional product paths must point to explicit file-hash completion receipts.
    'products': {},
}
SELECTION.parent.mkdir(parents=True, exist_ok=True)
if SELECTION.exists() and json.loads(SELECTION.read_text()) != settings:
    raise RuntimeError('Saved source selection differs. Inspect it before selecting another plan name.')
SELECTION.write_text(json.dumps(settings, indent=2) + '\n')
run_stage0('prepare', '--selection', SELECTION, '--name', STAGE0_NAME)


## 3. Audit or resume
Verify selected config, seal and adapter payloads; compare actual zero-initialization tensors. Sealed source units are reused. No optimizer-state replay, model download or inference.

In [ ]:
run_stage0('audit', '--config', PLAN)


## 4. Restore saved status
Safe in a fresh kernel after section 1. Reports audit evidence, missing prerequisites and the cumulative 20-unit budget. Unknown cost is not zero cost.

In [ ]:
run_stage0('status', '--config', PLAN)


## Next stage
Ticket 02 adds the CPU spectral cache. GPU qualification and production commands arrive in later tickets. Keep GPU disconnected for this slice.
Budget observations are entered through the public budget-record command described in the guide; no initial zero is assumed.